# OpenFold3 Training Quickstart — single H100

Getting an [OpenFold3](https://github.com/aqlaboratory/openfold-3) training run to complete on
one GPU, end to end: environment, a small real dataset, the config that matters, the launch,
and how to read what came out.

This is the **single-GPU** rung of the ladder. The same config scales to multi-GPU and
multi-node by changing two numbers — covered at the end.

> Every number below came from executing this notebook on a single
> [NVIDIA H100 80GB HBM3](https://www.nvidia.com/en-us/data-center/h100/).

**The short version:** training OpenFold3 is far lighter than it sounds. The dataset for this
run is 73 MB, peak GPU memory is 28 GB, and two epochs finish in under three minutes.

## What you need first

This notebook drives an existing OpenFold3 checkout — it does not install one. Two steps in a
shell before you start:

```bash
git clone https://github.com/aqlaboratory/openfold-3.git
cd openfold-3
pixi install -e openfold3-cuda12
```

The pixi environment is about 14 GB and needs [pixi](https://pixi.sh) 0.80 or newer, which the
repo's `pixi.toml` requires. Everything after this runs through `pixi run -e openfold3-cuda12`,
so this notebook's own kernel only needs to be able to start subprocesses — with one exception
noted in section 5.

Point the next cell at your checkout.

In [1]:
import os
import subprocess
from pathlib import Path

# EDIT THIS: your openfold-3 checkout.
OPENFOLD3_DIR = Path(os.environ.get("OPENFOLD3_DIR", Path.home() / "openfold-3"))
PIXI_ENV = "openfold3-cuda12"


def of3(*args, timeout=None):
    '''Run a command inside the pixi environment, from the checkout.

    WANDB_MODE=offline is essential, not decorative -- see section 3.'''
    result = subprocess.run(
        ["pixi", "run", "-e", PIXI_ENV, *args],
        cwd=OPENFOLD3_DIR, capture_output=True, text=True, timeout=timeout,
        env={**os.environ, "WANDB_MODE": "offline"},
    )
    return result


assert OPENFOLD3_DIR.exists(), f"no checkout at {OPENFOLD3_DIR}"
print(f"checkout: {OPENFOLD3_DIR}")

checkout: /home/ubuntu/jianwen/openfold-3


## 1. Environment

The kernels are architecture-specific, so confirm what you actually have before anything else.

In [2]:
probe = of3("python", "-c", '''
import torch
from importlib.metadata import version
cap = torch.cuda.get_device_capability(0)
print(f"openfold3   {version('openfold3')}")
print(f"torch       {torch.__version__}  (CUDA {torch.version.cuda})")
print(f"GPU         {torch.cuda.get_device_name(0)}  (sm_{cap[0]}{cap[1]})")
print(f"GPU memory  {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
''', timeout=600)
print(probe.stdout.strip())

openfold3   0.5.1.dev55+g3a9ff907f
torch       2.10.0  (CUDA 12.9)
GPU         NVIDIA H100 80GB HBM3  (sm_90)
GPU memory  79 GiB


## 2. The training subset

Full OpenFold3 training pulls a large PDB set from S3. The repo also ships a helper that samples
a handful of structures and downloads only those — which is all a smoke test needs:

```bash
pixi run -e openfold3-cuda12 setup-pdb-subset
```

It chains two scripts: `generate-pdb-subset` samples structure IDs from the full index, and
`download-pdb-subset` fetches just those. Defaults are 8 training and 4 validation structures;
`-- --train-size N` changes that.

The first run downloads a 1.6 GB index (`training_cache_with_templates.json`) to sample from.
That file is *not* training data — once the subset exists you can delete it.

In [3]:
subset = of3("setup-pdb-subset", timeout=7200)
tail = [l for l in subset.stdout.strip().split("\n") if l.strip()][-3:]
print("\n".join(tail))

datasets = OPENFOLD3_DIR / "datasets"
structures = list((datasets / "pdb_training_set").rglob("*.npz"))


def size_of(path):
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return total / 2**20


print(f"\nstructures (.npz)        {len(structures)}")
print(f"training data on disk    {size_of(datasets / 'pdb_training_set'):.0f} MB")
print(f"sampling index (delete)  {(datasets / 'training_cache_with_templates.json').stat().st_size / 2**30:.1f} GB")

  Progress: 48/48 (downloaded=0, skipped=48, failed=0)
Reference mols: 0 downloaded, 48 skipped, 0 failed
Done: 0 downloaded, 435 skipped, 1 failed

structures (.npz)        387
training data on disk    71 MB
sampling index (delete)  1.6 GB


### A path trap worth knowing

The repo contains `scripts/datasets/train_pdb_subset.yaml`, which looks like the config to use.
It isn't — it is a committed artifact still holding the absolute paths of whoever generated it
(`/home/jnwei/workspace/...`). Point `run_openfold` at it and every dataset path is wrong.

The generator writes a **fresh** copy, with your paths, to `datasets/train_pdb_subset.yaml` at
the repo root. That one is the real config.

In [4]:
import yaml

generated = datasets / "train_pdb_subset.yaml"
committed = OPENFOLD3_DIR / "scripts" / "datasets" / "train_pdb_subset.yaml"

for label, path in (("generated (use this)", generated), ("committed (a template)", committed)):
    cfg = yaml.safe_load(path.read_text())
    cache = cfg["dataset_paths"]["weighted-pdb"]["dataset_cache_file"]
    print(f"{label:24s} {path.relative_to(OPENFOLD3_DIR)}")
    print(f"{'':24s}   cache -> {cache}")

generated (use this)     datasets/train_pdb_subset.yaml
                           cache -> /home/ubuntu/jianwen/openfold-3/datasets/training_cache_with_templates_subset_8.json
committed (a template)   scripts/datasets/train_pdb_subset.yaml
                           cache -> /home/jnwei/workspace/openfold-3/scripts/datasets/training_cache_with_templates_subset_8.json


## 3. The training config

The generated YAML is a complete single-GPU training setup. The settings that decide whether it
runs at all, and how big it gets:

| Setting | Value | What it controls |
|---|---|---|
| `pl_trainer_args.devices` | `1` | GPUs per node — this is the single-GPU knob |
| `pl_trainer_args.num_nodes` | `1` | Nodes |
| `pl_trainer_args.precision` | `bf16-mixed` | Mixed precision; the memory figure below depends on it |
| `pl_trainer_args.max_epochs` | `2` | How long the smoke test runs |
| `data_module_args.batch_size` | `1` | Structures per step |
| `data_module_args.epoch_len` | `32` | Samples drawn per epoch |
| `dataset_configs...token_budget` | `384` | Crop size — the main memory lever |

There is one gap to close before running. `LoggingConfig` accepts Weights & Biases or nothing,
and the generated config sets `wandb_config: null` — so every loss the model computes is
discarded, and a successful run leaves an empty `logs/` directory beside a 6 GB checkpoint.

Two things are needed to capture them, and only one is obvious.

Setting `wandb_config.offline: true` in the YAML is **not sufficient on its own**. That flag
never reaches `wandb.init`, which still tries to log in and aborts the run before training
starts:

```
wandb.sdk.wandb_init ... maybe_login -> wandb_login._login(...)
```

What actually works is the environment variable `WANDB_MODE=offline`, which this notebook sets
for every subprocess in the `of3` helper above. Set both: the flag documents the intent, the
variable enforces it.

In [5]:
config = yaml.safe_load(generated.read_text())

config["logging_config"] = {
    "log_lr": True,
    "wandb_config": {
        "project": "openfold3-quickstart",
        "experiment_name": "single-h100",
        "offline": True,          # local metrics, no account, no network
    },
}
config["experiment_settings"]["output_dir"] = "./quickstart_run"

runner_yaml = datasets / "quickstart_single_h100.yaml"
runner_yaml.write_text(yaml.safe_dump(config, sort_keys=False))

print(yaml.safe_dump({
    "experiment_settings": config["experiment_settings"],
    "logging_config": config["logging_config"],
    "pl_trainer_args": config["pl_trainer_args"],
    "data_module_args": config["data_module_args"],
}, sort_keys=False).strip())

experiment_settings:
  mode: train
  output_dir: ./quickstart_run
  seed: 42
  restart_checkpoint_path: last
logging_config:
  log_lr: true
  wandb_config:
    project: openfold3-quickstart
    experiment_name: single-h100
    offline: true
pl_trainer_args:
  devices: 1
  num_nodes: 1
  precision: bf16-mixed
  max_epochs: 2
  mpi_plugin: false
  deepspeed_config_path: null
data_module_args:
  batch_size: 1
  num_workers: 4
  epoch_len: 4


## 4. Run it

```bash
pixi run -e openfold3-cuda12 run_openfold train \
    --runner-yaml datasets/quickstart_single_h100.yaml --seed 42
```

Two epochs of four steps. Expect a few minutes, most of it validation.

In [6]:
import threading
import time

peak_memory_mib = 0
sampling = True


def sample_gpu_memory():
    # Device-wide usage, so this is only meaningful on an otherwise idle GPU.
    global peak_memory_mib
    while sampling:
        probe = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True,
        )
        try:
            peak_memory_mib = max(peak_memory_mib, int(probe.stdout.strip().split("\n")[0]))
        except (ValueError, IndexError):
            pass
        time.sleep(2)


sampler = threading.Thread(target=sample_gpu_memory, daemon=True)
sampler.start()

start = time.perf_counter()
run = of3("run_openfold", "train",
          "--runner-yaml", str(runner_yaml.relative_to(OPENFOLD3_DIR)),
          "--seed", "42", timeout=7200)
elapsed = time.perf_counter() - start

sampling = False
sampler.join(timeout=5)

log = run.stdout + run.stderr
if run.returncode != 0:
    raise RuntimeError(
        "training failed -- last lines:\n"
        + "\n".join((run.stdout + run.stderr).strip().split("\n")[-15:])
    )

print(f"exit code   {run.returncode}")
print(f"wall time   {elapsed / 60:.1f} min")

progress = [l for l in log.replace("\r", "\n").split("\n") if "Epoch 1: 100%" in l]
print(f"last line   {progress[-1].strip() if progress else '(none)'}")
print(f"completed   {'Trainer.fit` stopped' in log}")
print(f"peak GPU    {peak_memory_mib / 1024:.1f} GiB")

exit code   0
wall time   4.6 min
last line   Epoch 1: 100%|██████████| 4/4 [01:24<00:00, 21.08s/it, v_num=7h2e]
completed   True
peak GPU    28.3 GiB


## 5. Read the results

Everything lands under the `output_dir` from the config. The layout has a wrinkle: enabling
wandb moves the checkpoint from `<output_dir>/checkpoints/` down to
`<output_dir>/<project>/<run id>/checkpoints/`, so a hardcoded path breaks when you turn
logging on.

Peak GPU memory is sampled with `nvidia-smi` during the run rather than read back from wandb.
wandb does record it, as `gpu.0.memoryAllocatedBytes`, but it flushes system stats when the run
closes -- read the file immediately afterwards and the field is silently absent, which reports a
confident zero.

The loss metrics live in a binary `run-*.wandb` file and nothing else holds them. Reading it
needs the `wandb` package — the one place this notebook's kernel must itself be the pixi
environment rather than just launching it. The cell below converts them to a `metrics.csv` you
can open with anything.

In [7]:
import csv
import glob

from wandb.proto import wandb_internal_pb2 as pb
from wandb.sdk.internal import datastore

run_dir = OPENFOLD3_DIR / "quickstart_run"
wandb_file = glob.glob(str(run_dir / "wandb" / "offline-run-*" / "run-*.wandb"))[0]

store = datastore.DataStore()
store.open_for_scan(wandb_file)

history = []
while True:
    data = store.scan_data()
    if data is None:
        break
    record = pb.Record()
    record.ParseFromString(data)
    kind = record.WhichOneof("record_type")
    if kind == "history":
        row = {}
        for item in record.history.item:
            key = item.key or ".".join(item.nested_key)
            try:
                row[key] = json.loads(item.value_json)
            except Exception:
                row[key] = item.value_json
        if "train/loss" in row:
            history.append(row)

columns = ["_step", "train/loss", "train/distogram_loss", "train/diffusion_loss"]
csv_path = run_dir / "metrics.csv"
with open(csv_path, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(history)

def as_number(value):
    '''wandb values arrive as JSON; anything non-numeric prints as a dash.'''
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


print(f"{'step':>5}{'loss':>10}{'distogram':>12}{'diffusion':>12}")
print("-" * 39)
for row in history:
    line = f"{str(row.get('_step', '?')):>5}"
    for key, width in (("train/loss", 10), ("train/distogram_loss", 12), ("train/diffusion_loss", 12)):
        value = as_number(row.get(key))
        line += f"{value:>{width}.3f}" if value is not None else f"{'-':>{width}}"
    print(line)

checkpoint = next(run_dir.rglob("last.ckpt"))
print(f"\npeak GPU memory   {peak_memory_mib / 1024:.1f} GiB (sampled during the run)")
print(f"checkpoint        {checkpoint.stat().st_size / 2**30:.1f} GiB")
print(f"metrics written   {csv_path.relative_to(OPENFOLD3_DIR)}")

 step      loss   distogram   diffusion
---------------------------------------
    0     6.739       4.159       6.613
    1     6.844       4.159       6.718
    2     7.255       4.158       7.129
    3     7.242       4.157       7.116
    9     7.159       4.155       7.033
   10     7.198       4.151       7.072
   11     6.697       4.149       6.571
   12     8.066       4.140       7.941

peak GPU memory   28.3 GiB (sampled during the run)
checkpoint        6.3 GiB
metrics written   quickstart_run/metrics.csv


### What these numbers do and don't say

The loss does not fall, and it should not be read as if it might. Eight optimiser steps over
eight structures is a smoke test: it proves the data pipeline, the model, the loss, the backward
pass and the checkpoint writer all work together. Learning anything would take orders of
magnitude more steps and data. A quickstart that showed a tidy descending curve here would be
showing you noise.

What the numbers do establish is capacity. Peak GPU memory at `token_budget: 384` sits around
28 GiB, comfortably inside one 80 GB H100 — and well inside a 40 GB card. Raise the token budget
and that is the number that moves first.

Two costs to plan for: each run writes a checkpoint of about 6 GB, and the environment is 14 GB.

## 6. Scaling up

The same YAML covers every rung; only two fields change.

| | `devices` | `num_nodes` |
|---|---|---|
| This notebook | 1 | 1 |
| One full node | 8 | 1 |
| Multi-node | 8 | 32 |

The launch command does not change. Beyond that, real training swaps the sampled subset for the
full PDB training set:

```bash
aws s3 sync s3://openfold3-data/pdb_training_set/ /shared/openfold3/pdb_training_set/ --no-sign-request
```

and starts from one of the configs in `examples/training_yamls/` — `initial_training.yml`
through `finetune_3.yml` — which carry the real schedules rather than a two-epoch smoke test.

## Links

- OpenFold3: <https://github.com/aqlaboratory/openfold-3>
- Training guide: <https://openfold-3.readthedocs.io/en/latest/training.html>
- Configuration reference: <https://openfold-3.readthedocs.io/en/latest/configuration_reference.html>
- Datasets: <https://portal.openfold.omsf.io/datasets>